# PropNet: Graph-Based Misinformation Detection

End-to-end demo using the UPFD dataset (PolitiFact + GossipCop).  
We sample **20% of each split** so the full notebook runs in a few minutes on a GPU.

**Models compared**
1. TextOnly\_BERT — logistic regression on mean BERT node features  
2. RF\_motif — random forest on 7 graph-structural motif features  
3. GraphOnly\_GAT — GAT on node features, no motifs  
4. Hybrid\_GAT — same GAT, BERT features present  
5. PropNet\_Fusion — GAT + attention-gated motif fusion  

## 0. Setup

In [ ]:
import os, sys, warnings
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from src.upfd_loader import load_upfd_graphs
from src.motifs import attach_motif_features, fit_motif_scaler, apply_motif_scaler, MOTIF_DIM, MOTIF_FEATURE_NAMES
from src.pooled_dataset import concat_datasets, stratified_pooled_split
from src.train import TrainingConfig, train_gnn
from src.utils import set_seed, compute_classification_metrics, plot_confusion, plot_roc

set_seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    # RTX 4060 perf knobs: TF32 matmul, cuDNN autotune, high-precision matmul
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision('high')
    except Exception:
        pass
    torch.cuda.empty_cache()
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('CUDA not available — running on CPU')
print(f'Device: {device}')

## 1. Dataset Paths

In [ ]:
POL_ROOT = os.path.join(REPO_ROOT, 'datasets', 'politifact')
GOS_ROOT = os.path.join(REPO_ROOT, 'datasets', 'gossipcop')

print('PolitiFact:', POL_ROOT, '— exists:', os.path.isdir(POL_ROOT))
print('GossipCop :', GOS_ROOT, '— exists:', os.path.isdir(GOS_ROOT))

## 2. Load UPFD Graphs

In [ ]:
pol_graphs = load_upfd_graphs(POL_ROOT, dataset_id='upfd_pol')
gos_graphs = load_upfd_graphs(GOS_ROOT, dataset_id='upfd_gos')

print(f'PolitiFact graphs: {len(pol_graphs)}')
print(f'GossipCop  graphs: {len(gos_graphs)}')
print(f'Node feature dim : {pol_graphs[0].x.shape[1]}')

pol_labels = [int(g.y.item()) for g in pol_graphs]
gos_labels = [int(g.y.item()) for g in gos_graphs]
print(f'PolitiFact label dist — real: {pol_labels.count(0)}, fake: {pol_labels.count(1)}')
print(f'GossipCop  label dist — real: {gos_labels.count(0)}, fake: {gos_labels.count(1)}')

## 3. Sample 20% and Split

In [ ]:
from sklearn.model_selection import train_test_split

SAMPLE_FRAC = 0.20
RNG = 42

def stratified_sample(graphs, frac, seed):
    labels = [int(g.y.item()) for g in graphs]
    idx = list(range(len(graphs)))
    sampled, _ = train_test_split(idx, train_size=frac, stratify=labels, random_state=seed)
    return [graphs[i] for i in sampled]

pol_sample = stratified_sample(pol_graphs, SAMPLE_FRAC, RNG)
gos_sample = stratified_sample(gos_graphs, SAMPLE_FRAC, RNG)

print(f'Sampled PolitiFact: {len(pol_sample)}')
print(f'Sampled GossipCop : {len(gos_sample)}')

all_graphs = concat_datasets(pol_sample, gos_sample)
print(f'Total pooled       : {len(all_graphs)}')

train_graphs, val_graphs, test_graphs = stratified_pooled_split(
    all_graphs, test_size=0.20, val_size=0.10, random_state=RNG
)
print(f'Split — train: {len(train_graphs)}, val: {len(val_graphs)}, test: {len(test_graphs)}')

## 4. Motif Feature Extraction

In [ ]:
from src.motifs import compute_motif_features

def attach_motif_features_fast(graphs):
    # NetworkX work is GIL-bound; serial list-comp beats thread overhead for small graphs.
    for g in graphs:
        g.motif_feat = torch.from_numpy(compute_motif_features(g)).unsqueeze(0)

print('Computing motif features...')
attach_motif_features_fast(train_graphs)
attach_motif_features_fast(val_graphs)
attach_motif_features_fast(test_graphs)

motif_mean, motif_std = fit_motif_scaler(train_graphs)
apply_motif_scaler(train_graphs, motif_mean, motif_std)
apply_motif_scaler(val_graphs,   motif_mean, motif_std)
apply_motif_scaler(test_graphs,  motif_mean, motif_std)

sample_motif = train_graphs[0].motif_feat.squeeze(0).numpy()
motif_df = pd.DataFrame({'feature': MOTIF_FEATURE_NAMES, 'value (train[0])': sample_motif})
print(motif_df.to_string(index=False))

## 5. Baseline — TextOnly BERT (logistic regression on mean node features)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

def mean_node_features(graphs):
    return np.stack([g.x.mean(dim=0).numpy() for g in graphs])

X_tr = mean_node_features(train_graphs)
X_va = mean_node_features(val_graphs)
X_te = mean_node_features(test_graphs)
y_tr = [int(g.y.item()) for g in train_graphs]
y_te = [int(g.y.item()) for g in test_graphs]

scaler_text = StandardScaler().fit(X_tr)
X_tr_s = scaler_text.transform(X_tr)
X_te_s = scaler_text.transform(X_te)

lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RNG)
lr.fit(X_tr_s, y_tr)
text_pred = lr.predict(X_te_s)
text_prob = lr.predict_proba(X_te_s)[:, 1].tolist()

text_metrics = compute_classification_metrics(y_te, text_pred.tolist(), text_prob)
print('TextOnly_BERT — ROC-AUC:', round(text_metrics['roc_auc'], 4),
      ' F1-macro:', round(text_metrics['f1_macro'], 4))

## 6. Baseline — RF Motif (random forest on structural features)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

def motif_matrix(graphs):
    return np.stack([g.motif_feat.squeeze(0).numpy() for g in graphs])

M_tr = motif_matrix(train_graphs)
M_te = motif_matrix(test_graphs)

rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=RNG, n_jobs=-1)
rf.fit(M_tr, y_tr)
rf_pred = rf.predict(M_te)
rf_prob = rf.predict_proba(M_te)[:, 1].tolist()

rf_metrics = compute_classification_metrics(y_te, rf_pred.tolist(), rf_prob)
print('RF_motif — ROC-AUC:', round(rf_metrics['roc_auc'], 4),
      ' F1-macro:', round(rf_metrics['f1_macro'], 4))

feat_imp = pd.DataFrame({'feature': MOTIF_FEATURE_NAMES, 'importance': rf.feature_importances_})
feat_imp.sort_values('importance', ascending=False).reset_index(drop=True)

## 7. GNN Training Config

In [ ]:
import tempfile

INPUT_DIM = train_graphs[0].x.shape[1]
_tmp_dir  = tempfile.mkdtemp(prefix='propnet_demo_')

# Tuned for RTX 4060 + 20% sample: bigger batch (small graphs fit easily),
# fewer epochs and tighter patience (small data converges fast).
BASE_CFG = dict(
    batch_size=512,
    hidden_dim=128,
    dropout=0.4,
    lr=1e-3,
    weight_decay=5e-4,
    max_epochs=40,
    patience=6,
    lr_scheduler_patience=3,
    lr_scheduler_factor=0.5,
    grad_clip_norm=1.0,
    use_amp=True,
    device=device,
)
print('Input dim:', INPUT_DIM, '| device:', device, '| batch_size:', BASE_CFG['batch_size'])

## 8. GraphOnly GAT

In [ ]:
cfg_go = TrainingConfig(
    **BASE_CFG,
    checkpoint_path=os.path.join(_tmp_dir, 'demo_graphonly.pt'),
)

go_result = train_gnn(train_graphs, val_graphs, test_graphs, INPUT_DIM, cfg_go)
print(f'GraphOnly_GAT — best epoch: {go_result["best_epoch"]}  '
      f'ROC-AUC: {go_result["metrics"]["roc_auc"]:.4f}  '
      f'F1-macro: {go_result["metrics"]["f1_macro"]:.4f}')

## 9. Hybrid GAT (BERT node features, no motif fusion)

In [ ]:
# Hybrid_GAT uses the same GATGraphClassifier architecture, same hyperparams,
# and same training graphs as GraphOnly_GAT — the only conceptual difference is
# framing (BERT features are *present* in g.x either way). Reuse the trained
# result instead of repeating an identical run.
hy_result = go_result
print(f'Hybrid_GAT — best epoch: {hy_result["best_epoch"]}  '
      f'ROC-AUC: {hy_result["metrics"]["roc_auc"]:.4f}  '
      f'F1-macro: {hy_result["metrics"]["f1_macro"]:.4f}  '
      f'(reused from GraphOnly_GAT — identical architecture & inputs)')

## 10. PropNet Fusion (GAT + attention-gated motif)

In [ ]:
cfg_fu = TrainingConfig(
    **BASE_CFG,
    use_attention_fusion=True,
    motif_dim=MOTIF_DIM,
    checkpoint_path=os.path.join(_tmp_dir, 'demo_fusion.pt'),
)

fu_result = train_gnn(train_graphs, val_graphs, test_graphs, INPUT_DIM, cfg_fu)
print(f'PropNet_Fusion — best epoch: {fu_result["best_epoch"]}  '
      f'ROC-AUC: {fu_result["metrics"]["roc_auc"]:.4f}  '
      f'F1-macro: {fu_result["metrics"]["f1_macro"]:.4f}')

## 11. Results Comparison

In [ ]:
rows = [
    {'model': 'TextOnly_BERT',    **{k: round(v, 4) for k, v in text_metrics.items()}},
    {'model': 'RF_motif',         **{k: round(v, 4) for k, v in rf_metrics.items()}},
    {'model': 'GraphOnly_GAT',    **{k: round(v, 4) for k, v in go_result['metrics'].items()}},
    {'model': 'Hybrid_GAT',       **{k: round(v, 4) for k, v in hy_result['metrics'].items()}},
    {'model': 'PropNet_Fusion',   **{k: round(v, 4) for k, v in fu_result['metrics'].items()}},
]

results_df = pd.DataFrame(rows).sort_values('roc_auc', ascending=False).reset_index(drop=True)
display(results_df[['model', 'roc_auc', 'f1_macro', 'accuracy', 'precision_fake', 'recall_fake']])

## 12. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, result, name in zip(axes,
                             [go_result, hy_result, fu_result],
                             ['GraphOnly_GAT', 'Hybrid_GAT', 'PropNet_Fusion']):
    hist = result['history']
    ax.plot(hist['epoch'], hist['train_loss'], label='train loss')
    ax.plot(hist['epoch'], hist['val_roc_auc'], label='val AUC')
    ax.axvline(result['best_epoch'], color='red', linestyle='--', alpha=0.5, label=f'best={result["best_epoch"]}')
    ax.set_title(name)
    ax.set_xlabel('Epoch')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 13. Confusion Matrices & ROC Curves

In [ ]:
plot_confusion(fu_result['y_true'], fu_result['y_pred'], title='PropNet Fusion — Confusion Matrix')
plot_roc(fu_result['y_true'], fu_result['y_prob'], title='PropNet Fusion — ROC Curve')

plot_confusion(hy_result['y_true'], hy_result['y_pred'], title='Hybrid GAT — Confusion Matrix')
plot_roc(hy_result['y_true'], hy_result['y_prob'], title='Hybrid GAT — ROC Curve')

## 14. ROC Comparison — All Models

In [ ]:
from sklearn.metrics import roc_curve, auc

fig, ax = plt.subplots(figsize=(7, 5))

for name, yt, yp in [
    ('TextOnly_BERT',  y_te,               text_prob),
    ('RF_motif',       y_te,               rf_prob),
    ('GraphOnly_GAT',  go_result['y_true'], go_result['y_prob']),
    ('Hybrid_GAT',     hy_result['y_true'], hy_result['y_prob']),
    ('PropNet_Fusion', fu_result['y_true'], fu_result['y_prob']),
]:
    fpr, tpr, _ = roc_curve(yt, yp)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc(fpr,tpr):.3f})')

ax.plot([0,1],[0,1],'k--', alpha=0.4)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC — All Models (20% UPFD sample)')
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()